# Agent 시스템 평가(Evaluation)

## 학습 목표

- agent workflow를 왜 구조적으로 평가해야 하는지 이해한다.
- 균형 잡힌 40문항 evaluation dataset을 직접 살펴본다.
- correctness, retrieval, grounding, abstention, latency, reasoning step 수를 해석한다.
- baseline과 agent workflow를 표와 차트로 비교한다.


## 개념 설명

평가는 실행 환경이 흔들리면 의미가 없다. 이 셀은 현재 interpreter와 runtime profile을 출력해서, 로컬 환경인지 DGX 환경인지 빠르게 sanity check할 수 있게 해준다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## 왜 evaluation이 중요한가

한두 개의 인상적인 데모만으로는 아키텍처의 신뢰성을 말할 수 없다. evaluation은 설계 가설을 측정 가능한 근거로 바꿔준다. 이 저장소에서 baseline workflow가 중요한 이유도 여기에 있다. 모든 후속 개선을 비교할 기준선(control) 역할을 하기 때문이다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.evaluator import load_eval_dataset, run_evaluation_suite

dataset = load_eval_dataset()
dataset_frame = pd.DataFrame(dataset)
print(f'Total questions: {len(dataset_frame)}')
dataset_frame[['id', 'question_type', 'expected_status', 'question']].head(12)

## 평가 데이터셋 설계(Designing evaluation datasets)

좋은 evaluation set은 단순히 크기만 큰 데이터셋이 아니다. failure pattern이 읽히도록 구성도 균형 있어야 한다. 지금의 확장된 데이터셋은 query type마다 8문항씩 맞춰져 있어서 lookup, comparison, summary, multi-hop, insufficient-evidence 케이스를 공정하게 비교할 수 있다.


## 구현

이 평가 파이프라인은 `src/evaluator.py`에 구현되어 있다. 여기서는 먼저 데이터셋 분포를 확인하고, 그다음 baseline과 agent workflow를 반복 실행한 뒤, 마지막으로 지표 간 trade-off를 시각화한다.


In [ ]:
distribution = dataset_frame['question_type'].value_counts().sort_index()
ax = distribution.plot(kind='bar', color='#4C78A8', title='Evaluation Dataset Distribution by Query Type')
ax.set_xlabel('query_type')
ax.set_ylabel('question count')
plt.tight_layout()
plt.show()
distribution.reset_index().rename(columns={'index': 'question_type', 'question_type': 'count'})

## 지표(metrics)

내장 evaluation pipeline은 baseline과 agent workflow를 반복 실행하면서 answer correctness, retrieval hit rate, grounding pass rate, abstain precision, latency, average reasoning steps를 계산한다. 결과를 파일로 저장해두면 다음 노트북에서 그대로 재사용할 수 있다.


In [ ]:
results, summary = run_evaluation_suite(repeats=2, persist_outputs=True)
summary

## baseline과 agent 비교

단일 summary table도 의미가 있지만, query type별 breakdown을 같이 봐야 해석이 쉬워진다. 아래 셀은 시스템 단위 요약과 query type 단위 평균을 함께 보여주므로, agent workflow가 어디에서 가장 이득을 주는지 읽을 수 있다.


In [ ]:
question_type_breakdown = (
    results.groupby(['system', 'expected_question_type'])[[
        'answer_correctness',
        'retrieval_hit_rate',
        'grounding_pass_rate',
        'abstain_precision',
        'latency_seconds',
        'average_steps',
    ]]
    .mean()
    .round(3)
)

display(summary)
display(question_type_breakdown)

## 실험

radar chart는 여러 metric 사이의 trade-off를 한 번에 보여주기에 교육용으로 좋다. 여기서는 answer correctness, retrieval hit rate, grounding pass rate, abstain precision, 그리고 latency를 반영한 speed score를 함께 비교한다.


In [ ]:
radar = summary.set_index('system')[['answer_correctness', 'retrieval_hit_rate', 'grounding_pass_rate', 'abstain_precision']].copy()
radar['speed_score'] = 1.0 / summary.set_index('system')['latency'].clip(lower=0.001)
radar['speed_score'] = radar['speed_score'] / radar['speed_score'].max()
radar_metrics = list(radar.columns)
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
for system, row in radar.iterrows():
    values = row.tolist()
    values += values[:1]
    ax.plot(angles, values, label=system)
    ax.fill(angles, values, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics)
ax.set_title('Baseline vs Agent Workflow Radar View')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()
radar.round(3)

## 결과 해석

여기서 봐야 할 핵심 trade-off는 agent workflow가 더 많은 step과 때로는 더 큰 latency를 들이면서도, grounding과 abstention 품질을 더 높인다는 점이다. 내부 assistant나 research workflow에서는 이 trade-off가 충분히 가치 있다.


In [ ]:
metric_deltas = summary.set_index('system').loc['agent_workflow'] - summary.set_index('system').loc['baseline']
metric_deltas.to_frame(name='agent_minus_baseline').round(3)

## 핵심 정리

- 이 실험을 통해 evaluation은 단순 데모를 **측정 가능한 시스템 주장**으로 바꾼다는 점을 확인했다.
- 균형 잡힌 dataset이 있어야 어떤 query type에서 약한지 명확히 읽힌다.
- agent workflow는 보통 더 많은 단계와 비용을 쓰지만, 그 대가로 더 강한 근거 검증(grounding verification)과 abstention 품질을 얻는다.
- 면접에서는 "무엇을 metric으로 봤는가"라는 질문에 대해 **correctness, retrieval, grounding, abstention, latency, steps**를 함께 본 이유를 설명하면 좋다.
